# Module 6: Generative AI - Introduction to AI Agents

An **AI Agent** is an autonomous system powered by a Large Language Model that can plan, reason, use external **tools** (e.g., search engines, calculators, database querying), and interact with its environment to achieve a specific goal.

In this notebook, we will explore:
1. **Agent Architecture**: Planning, memory, and tools.
2. **The ReAct Paradigm**: Alternating between **Reasoning** (Thought) and **Action** (calling tools).
3. **Building an Agent Loop from Scratch**: Creating Python functions as tools and writing a loop that executes them based on the LLM's instructions.

> [!NOTE]
> **No API Key Required**: We use a **Simulated LLM agent parser** to demonstrate the ReAct framework in action without requiring external API keys. You can plug in a real LLM API (like OpenAI or Gemini) easily.

In [ ]:
import re

---

## 1. Defining Tools

Tools are Python functions that our agent can choose to execute. Let's write two simple tools:
1. **calculate**: Evaluates math expressions.
2. **get_weather**: Looks up mock weather degrees for a city.

In [ ]:
def calculate(expression):
    """Evaluates mathematical calculations. Input must be a valid math string e.g., '2 + 2' or '5 * 10'."""
    try:
        # Use a safe eval alternative for learning purposes
        # In production, use standard arithmetic parsers to prevent code execution hazards
        clean_expr = re.sub(r'[^0-9+\-*/().\s]', '', expression)
        return str(eval(clean_expr))
    except Exception as e:
        return f"Error evaluating math: {str(e)}"

def get_weather(city):
    """Retrieves current temperature for a city. Input must be a city name string."""
    mock_db = {
        'new york': '18°C', 
        'london': '12°C', 
        'tokyo': '22°C', 
        'paris': '15°C'
    }
    city_clean = city.strip().lower()
    return mock_db.get(city_clean, "Unknown temperature for this city.")

# Tool Directory
available_tools = {
    'calculate': calculate,
    'get_weather': get_weather
}

---

## 2. ReAct System Prompt

We write a system prompt instruction instructing the LLM to follow the **ReAct format**:
```text
Thought: [Reasoning about the problem]
Action: [tool_name: argument]
Observation: [result returned by the tool]
... (repeat until solved)
Thought: I know the final answer.
Final Answer: [The response to the user]
```

In [ ]:
system_prompt = """You are an AI Agent with access to tools.


Let's write a python parser and loop that parses this format.

In [ ]:
# Simulated LLM ReAct response generator
# This acts as the LLM, simulating how it decides to use tools one-by-one
class SimulatedReActLLM:
    def __init__(self):
        self.step = 0
        
    def generate_response(self, prompt):
        # We simulate the LLM's thought process for a multi-step query:
        # "What is the temperature in New York multiplied by 2?"
        self.step += 1
        
        if self.step == 1:
            return """Thought: The user wants to know the temperature in New York multiplied by 2.
First, I need to get the current weather/temperature in New York.
Action: get_weather[New York]
"""
        elif self.step == 2:
            return """Thought: I received the observation that the temperature in New York is 18°C.
Now I need to multiply 18 by 2.
Action: calculate[18 * 2]
"""
        else:
            return """Thought: I received the calculation result 36.
I have all the information required to compile the final answer.
Final Answer: The temperature in New York is 18°C, and multiplying it by 2 yields 36.
"""

---

## 3. The Agent Execution Loop

The runner loop handles sending the prompt, extracting actions using Regex, calling the Python functions (tools), printing the observation, and feeding it back to the model.

In [ ]:
def run_agent(user_query):
    print(f"User Query: {user_query}\n")
    
    llm = SimulatedReActLLM()
    agent_prompt = f"{system_prompt}\n\nUser Query: {user_query}\n"
    
    max_iterations = 5
    
    for i in range(max_iterations):
        print(f"--- Agent Iteration {i+1} ---")
        
        # Get the LLM output
        response = llm.generate_response(agent_prompt)
        print(response.strip())
        
        # Append LLM response to prompt history
        agent_prompt += response
        
        # Check if LLM outputted Final Answer
        if "Final Answer:" in response:
            print("\nSuccess: Goal achieved!")
            break
            
        # Regex parser to extract Action: tool_name[argument]
        match = re.search(r'Action:\s*(\w+)\[(.*)\]', response)
        
        if match:
            tool_name = match.group(1)
            tool_arg = match.group(2)
            
            if tool_name in available_tools:
                print(f"\n>> [SYSTEM] Executing Tool: {tool_name}({tool_arg})")
                tool_output = available_tools[tool_name](tool_arg)
                print(f">> [SYSTEM] Observation result: {tool_output}\n")
                
                # Append Observation back to the prompt history
                agent_prompt += f"Observation: {tool_output}\n"
            else:
                print(f"\n>> [SYSTEM] Error: Tool '{tool_name}' is not available.\n")
                agent_prompt += f"Observation: Tool not found.\n"
        else:
            print("\n>> [SYSTEM] Error: No valid Action format found. Stopping.")
            break

run_agent("What is the temperature in New York multiplied by 2?")

---

## 🏋️ Exercise: Add a Google Search Tool

Add a mock search tool called `search_web` that returns search results for:
- "Who founded Apple?" -> "Steve Jobs, Steve Wozniak, and Ronald Wayne."
- "Capital of France" -> "Paris."

1. Write the `search_web` function.
2. Add it to the `available_tools` directory.
3. Try drawing up a plan of how a real LLM would solve: `"Who founded Apple and what is the current weather in Paris?"` utilizing multiple tools sequentially.

In [ ]:
# YOUR CODE HERE